# 01 - NumPy Foundations

**Study track:** NumPy & Pandas for a Java / SQL engineer.
**This notebook:** what an `ndarray` really is, how it differs from a Java array, dtypes, indexing, views vs copies, and the `axis` idea.

---

## Why NumPy exists

A Python `list` holds *pointers* to objects scattered in memory. Each `int` in a list is a full Python object (28 bytes) with a type header and a refcount.

A NumPy `ndarray` holds **raw values in one contiguous block of memory**, all of the same type. That is exactly what a Java `int[]` or `double[]` is.

| Concept | Java | Python list | NumPy ndarray |
|---|---|---|---|
| Memory layout | contiguous primitives | pointers to boxed objects | contiguous primitives |
| Element type | fixed at compile time (`int[]`) | anything, mixed | fixed `dtype` |
| Size | fixed | grows | fixed (resize = new array) |
| Loop cost | JIT-compiled, fast | interpreted, slow | done in C, fast |
| Multi-dimensional | `int[][]` = array of arrays | nested lists | one block + a `shape` |

The single most important sentence in this whole track:

> **Do not write loops over NumPy/Pandas data. Express the whole operation on the whole array at once.** That is called *vectorization*, and it is the equivalent of letting the database engine do a `SELECT` instead of pulling rows into your Java app and looping.

In [ ]:
import numpy as np
import sys

print("NumPy version:", np.__version__)

## 1. The memory difference, measured

Let us prove the claim above rather than just assert it.

In [ ]:
py_list = list(range(1_000_000))
np_arr  = np.arange(1_000_000)

# Rough memory footprint
list_bytes = sys.getsizeof(py_list) + sum(sys.getsizeof(x) for x in py_list[:1000]) / 1000 * 1_000_000
print("python list  ~", int(list_bytes) // 1024 // 1024, "MB")
print("numpy array   ", np_arr.nbytes // 1024 // 1024, "MB")

In [ ]:
import time, math

py_floats = [i * 0.5 for i in range(3_000_000)]
np_floats = np.arange(3_000_000) * 0.5

t0 = time.perf_counter(); r1 = [math.sqrt(x) * 2 + 1 for x in py_floats]; t1 = time.perf_counter()
t2 = time.perf_counter(); r2 = np.sqrt(np_floats) * 2 + 1;                t3 = time.perf_counter()

print("python list comprehension:", round(t1 - t0, 4), "s")
print("numpy vectorized         :", round(t3 - t2, 4), "s")
print("speedup                  :", round((t1 - t0) / (t3 - t2), 1), "x")

## 2. Creating arrays

Think of these as the static factory methods you would write in a Java `ArrayUtils` class.

In [ ]:
# From an existing Python sequence
a = np.array([1, 2, 3, 4])
print(a, a.dtype)

# 2-D: a list of lists becomes a matrix (NOT an array of arrays - see section 3)
m = np.array([[1, 2, 3],
              [4, 5, 6]])
print(m)
print("shape:", m.shape)

In [ ]:
print("arange   :", np.arange(0, 10, 2))          # like a for(int i=0;i<10;i+=2) loop
print("linspace :", np.linspace(0, 1, 5))         # 5 evenly spaced values, endpoints included
print("zeros    :", np.zeros(4))
print("ones     :", np.ones((2, 3), dtype=int))
print("full     :", np.full(3, 7.5))
print("eye      :\n", np.eye(3))                  # identity matrix
print("empty    :", np.empty(3).shape)            # uninitialised memory - fast, contents are garbage

In [ ]:
# Random data - always go through a Generator, not the old np.random.* functions
rng = np.random.default_rng(seed=42)     # seed makes results reproducible

print("uniform  :", rng.random(4).round(3))
print("normal   :", rng.normal(loc=0, scale=1, size=4).round(3))
print("integers :", rng.integers(low=1, high=7, size=10))   # like a dice roll, high is exclusive
print("choice   :", rng.choice(["A", "B", "C"], size=5))

> **Java note:** `np.random.default_rng(seed)` is the analogue of `new Random(seed)`. The older module-level `np.random.rand()` shares one global state - same problem as a static `Random` field, so avoid it in new code.

## 3. `shape`, `ndim`, and the crucial point about dimensions

In Java, a 2-D array `int[][] m` is genuinely an array of row references. Rows can have different lengths (jagged), and each row is a separate memory allocation.

In NumPy, a 2-D array is **one flat block of memory plus a shape**. It cannot be jagged. `shape=(2, 3)` means 6 values laid out end to end, interpreted as 2 rows of 3.

In [ ]:
m = np.array([[1, 2, 3],
              [4, 5, 6]])

print("ndim     :", m.ndim)        # number of dimensions (axes)
print("shape    :", m.shape)       # length of each dimension
print("size     :", m.size)        # total number of elements
print("dtype    :", m.dtype)       # element type
print("itemsize :", m.itemsize)    # bytes per element
print("nbytes   :", m.nbytes)      # total bytes = size * itemsize
print("flat     :", m.ravel())     # the underlying 1-D sequence

### Naming the axes

- **axis 0** runs *down* the rows (the vertical direction)
- **axis 1** runs *across* the columns (the horizontal direction)

A reliable rule: **`axis=k` is the axis that disappears.** `m.sum(axis=0)` on a `(2,3)` array removes axis 0 and leaves shape `(3,)`.

In [ ]:
print("m:\n", m)
print()
print("m.sum()         =", m.sum(),        " -> scalar, everything collapsed")
print("m.sum(axis=0)   =", m.sum(axis=0), " -> shape", m.sum(axis=0).shape, " (one value per column)")
print("m.sum(axis=1)   =", m.sum(axis=1), "     -> shape", m.sum(axis=1).shape, " (one value per row)")

**SQL analogy:** `m.sum(axis=0)` is `SELECT SUM(col) FROM t` for each column. `m.sum(axis=1)` is a per-row expression, like `SELECT c1 + c2 + c3 FROM t`.

## 4. dtypes - the type system

Every array has exactly one element type, chosen at creation and fixed afterwards.

| NumPy dtype | Bytes | Java equivalent | Range / note |
|---|---|---|---|
| `int8` / `uint8` | 1 | `byte` / - | -128..127 / 0..255 |
| `int16` / `uint16` | 2 | `short` / `char` | |
| `int32` / `uint32` | 4 | `int` | |
| `int64` / `uint64` | 8 | `long` | default integer on Linux/macOS |
| `float32` | 4 | `float` | ~7 significant digits |
| `float64` | 8 | `double` | default float, ~15 digits |
| `complex128` | 16 | - | |
| `bool_` | 1 | `boolean` | stored as a whole byte |
| `datetime64[ns]` | 8 | `Instant` | nanoseconds since 1970-01-01 |
| `timedelta64[ns]` | 8 | `Duration` | |
| `object` | 8 (pointer) | `Object` | escape hatch, slow - avoid |
| `str_` / `<U10` | 4 per char | `String` | fixed-width unicode |

In [ ]:
print(np.array([1, 2, 3]).dtype)            # int64 - inferred
print(np.array([1.0, 2, 3]).dtype)          # float64 - one float promotes everything
print(np.array([1, 2, 3], dtype=np.float32).dtype)
print(np.array([True, False]).dtype)
print(np.array(["abc", "de"]).dtype)        # <U3 = unicode, max 3 chars
print(np.array([1, "a", 2.0]).dtype)        # everything becomes a string

In [ ]:
# Casting: astype ALWAYS returns a new array (a copy)
x = np.array([1.7, -1.7, 2.9])
print("astype(int) truncates toward zero, like a Java (int) cast:", x.astype(np.int64))
print("np.round then cast                                        :", np.round(x).astype(np.int64))

### Overflow: NumPy wraps, it does not promote

This surprises people coming from Python (where `int` is arbitrary precision) but will feel familiar from Java, where `byte b = 127; b++` wraps to -128.

In [ ]:
small = np.array([120, 125], dtype=np.int8)
with np.errstate(over="ignore"):
    print("int8 120 + 10 =", small + np.int8(10))   # wraps around, exactly like Java byte arithmetic

print()
print("float32 precision loss:", np.float32(0.1) + np.float32(0.2))
print("float64               :", np.float64(0.1) + np.float64(0.2))
print("0.1 + 0.2 == 0.3      :", 0.1 + 0.2 == 0.3, " <- never compare floats with ==")
print("np.isclose            :", np.isclose(0.1 + 0.2, 0.3))

## 5. Indexing and slicing

1-D indexing is identical to Java. The interesting part starts at 2-D.

In [ ]:
a = np.arange(10) * 10
print(a)
print("a[0]     :", a[0])
print("a[-1]    :", a[-1])        # negative index counts from the end - no Java equivalent
print("a[2:5]   :", a[2:5])       # start inclusive, stop exclusive - same as List.subList
print("a[:3]    :", a[:3])
print("a[5:]    :", a[5:])
print("a[::2]   :", a[::2])       # every 2nd element
print("a[::-1]  :", a[::-1])      # reversed

In [ ]:
m = np.arange(1, 13).reshape(3, 4)
print(m)
print()
print("m[1, 2]    =", m[1, 2], "   <- row 1, col 2. In Java you would write m[1][2]")
print("m[1]       =", m[1], "   <- whole row")
print("m[:, 2]    =", m[:, 2], "  <- whole column. Java has no syntax for this at all")
print("m[0:2, 1:3]=\n", m[0:2, 1:3])

## 6. Views vs copies - the #1 source of bugs

A basic slice of a NumPy array is a **view**: a new array object that points at the *same memory*. Writing through the view changes the original.

This is like `List.subList()` in Java, which is also a live view, not `Arrays.copyOfRange()`, which copies.

In [ ]:
original = np.arange(10)
view = original[2:5]

view[0] = 999
print("original after writing to the view:", original)   # the 2 became 999
print("is it a view?", view.base is original)

In [ ]:
# Force a copy when you want isolation
original = np.arange(10)
copy = original[2:5].copy()
copy[0] = 999
print("original is untouched:", original)
print("copy.base:", copy.base)

### Rule of thumb

| Operation | Result |
|---|---|
| basic slice `a[2:5]`, `a[:, 1]` | **view** |
| `reshape`, `ravel`, `.T`, `transpose` | view when possible |
| fancy indexing `a[[0, 2, 4]]` | **copy** |
| boolean mask `a[a > 5]` | **copy** |
| `astype`, arithmetic `a + 1`, `flatten` | **copy** |

Quick check: `arr.base is not None` means it is a view of something.

In [ ]:
a = np.arange(10)
checks = {
    "a[2:5]          ": a[2:5],
    "a.reshape(2,5)  ": a.reshape(2, 5),
    "a[[0,2,4]]      ": a[[0, 2, 4]],
    "a[a > 5]        ": a[a > 5],
    "a.flatten()     ": a.flatten(),
    "a + 1           ": a + 1,
}
for name, r in checks.items():
    print(name, "-> view" if r.base is not None else "-> copy")

## 7. Reshaping

`reshape` does not move any data. It only changes how the same flat block is interpreted.

In [ ]:
a = np.arange(12)
print("reshape(3, 4):\n", a.reshape(3, 4))
print()
print("reshape(2, 6):\n", a.reshape(2, 6))
print()
print("reshape(3, -1) - the -1 means 'work it out for me':", a.reshape(3, -1).shape)

In [ ]:
m = np.arange(6).reshape(2, 3)
print("m:\n", m)
print("m.T (transpose, a view):\n", m.T)
print()
print("ravel   (view if possible):", m.ravel())
print("flatten (always a copy)  :", m.flatten())

In [ ]:
# Adding an axis - needed constantly for broadcasting (notebook 02)
v = np.array([1, 2, 3])
print("v shape          :", v.shape)
print("v[:, None] shape :", v[:, np.newaxis].shape, " <- column vector")
print(v[:, np.newaxis])
print("v[None, :] shape :", v[np.newaxis, :].shape, " <- row vector")

## 8. Joining and splitting

In [ ]:
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])

print("vstack / concatenate axis=0 (stack rows, like SQL UNION ALL):\n", np.vstack([a, b]))
print()
print("hstack / concatenate axis=1 (add columns, like a positional JOIN):\n", np.hstack([a, b]))
print()
print("stack - creates a NEW axis, shape:", np.stack([a, b]).shape)

In [ ]:
x = np.arange(12)
print("split into 3:", np.split(x, 3))
print("split at indices 2 and 7:", np.array_split(x, [2, 7]))

## 9. Exercises

Try each one before opening the solution cell below it.

**E1.** Create a 5x5 array where every value is its flat position (0..24), then print the middle 3x3 block.

**E2.** Given `temps_c = np.array([0, 37, 100, -40])`, convert to Fahrenheit (`F = C * 9/5 + 32`) without a loop.

**E3.** You have `a = np.arange(10)`. Write one line that sets every even-indexed element to 0 **without** affecting a separate backup copy you made first.

**E4.** Explain in one sentence why `np.arange(3).reshape(3, 1) + np.arange(3)` produces a 3x3 array. (Full answer in notebook 02.)

In [ ]:
# Solutions
g = np.arange(25).reshape(5, 5)
print("E1:\n", g[1:4, 1:4])
print()

temps_c = np.array([0, 37, 100, -40])
print("E2:", temps_c * 9 / 5 + 32)
print()

a = np.arange(10)
backup = a.copy()          # .copy() is the point - a plain slice would be a view
a[::2] = 0
print("E3: a =", a, " backup =", backup)
print()
print("E4: shapes (3,1) and (3,) are broadcast to (3,3) - the length-1 axis is stretched.")

## Recap

- `ndarray` = contiguous typed memory + a `shape`. Closest Java relative: `double[]` with extra metadata.
- One `dtype` per array, fixed width, wraps on overflow like Java primitives.
- `axis=k` is **the axis that disappears** in a reduction.
- Basic slices are **views** (shared memory). Fancy/boolean indexing and arithmetic produce **copies**.
- `reshape` is free; it never moves data.

**Next:** `02_numpy_operations.ipynb` - vectorization, broadcasting, boolean masks, and the operations you will actually use every day.